# Botanix — Swin-T: PlantCLEF Pre-train → Multi-Label Ürün Tespiti

**Mimari:** Swin Transformer Tiny (Swin-T)  
**PlantCLEF 2026:** https://www.imageclef.org/PlantCLEF2026  
**GPU:** H200 NVL (Vast.ai) — önerilen

## Çalışma Akışı

| Faz | Dataset | Görev | Kayıp Fonksiyonu | Açıklama |
|-----|---------|-------|-----------------|----------|
| **Faz 1** | PlantCLEF 2024 (~1.4M, 7806 tür) | Single-label | CrossEntropyLoss | Botanik alan bilgisi kazanımı |
| **Faz 2** | Kendi hastalık veri seti (105 sınıf) | **Multi-label** | BCEWithLogitsLoss | Ürün bazlı çoklu hastalık tespiti |

## Boyut Stratejisi

| Dataset | Orijinal Boyut | Model Girişi | Strateji |
|---------|---------------|-------------|----------|
| PlantCLEF | 800px (max side) | 224×224 | RandomResizedCrop — doğal çeşitlilik |
| Hastalık veri seti | 256px | 224×224 | Resize→224 + agresif augmentation |

> **Mobil Hedef:** Swin-T (~28M param, ~110MB) — TorchScript + INT8 quantization ile  
> mobil cihazda ~15-25ms inference süresi hedeflenmektedir.

In [ ]:
# ── Kurulum & Dataset ─────────────────────────────────────────────────────────
# PlantCLEF 2024 (~160GB, 800px versiyon) — Vast.ai terminalinde:
#   mkdir -p ../plantclef
#   wget -c "https://lab.plantnet.org/LifeCLEF/PlantCLEF2024/single_plant_training_data/\
#   PlantCLEF2024singleplanttrainingdata_800_max_side_size.tar" -P ../plantclef/
#   tar -xf ../plantclef/PlantCLEF2024singleplanttrainingdata_800_max_side_size.tar -C ../plantclef/
#   wget "https://lab.plantnet.org/LifeCLEF/PlantCLEF2024/single_plant_training_data/\
#   PlantCLEF2024singleplanttrainingdata.csv" -P ../plantclef/

!pip install -q kagglehub timm

import kagglehub, shutil, os
from pathlib import Path

PLANTCLEF_DIR = Path("../plantclef")
FINETUNE_DIR  = Path("../data")

# Hastalık veri setini indir
_raw = kagglehub.dataset_download("mertcangelbal/plant-leaf-disease-classification-dataset")
print("Hastalık dataset:", _raw)
if not FINETUNE_DIR.exists():
    shutil.copytree(_raw, str(FINETUNE_DIR))
    print(f"Kopyalandı: {FINETUNE_DIR}")
else:
    print(f"Mevcut: {FINETUNE_DIR}")

print(f"PlantCLEF mevcut: {PLANTCLEF_DIR.exists()}")
print(f"Fine-tune mevcut: {FINETUNE_DIR.exists()}")

In [ ]:
# ── Kütüphaneler ─────────────────────────────────────────────────────────────
import os, time, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image
import timm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, average_precision_score
)
from sklearn.preprocessing import label_binarize

print(f"PyTorch  : {torch.__version__}")
print(f"timm     : {timm.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Konfigürasyon ─────────────────────────────────────────────────────────────
CONFIG = {
    # Dizinler
    "plantclef_dir":      "../plantclef",
    "finetune_dir":       "../data",

    # Görüntü boyutları
    "pretrain_img_size":  224,   # PlantCLEF 800px → crop 224
    "finetune_img_size":  224,   # Hastalık 256px  → resize 224

    # Batch & worker
    "pretrain_batch":     64,    # H200 için 64 — RTX 5090 için 32'ye indir
    "finetune_batch":     32,
    "num_workers":        8,

    # Faz 1 — PlantCLEF Pre-train (single-label)
    "plantclef_classes": 7806,
    "pretrain_epochs":   10,
    "pretrain_lr":       3e-4,
    "pretrain_warmup":   2,      # warmup epoch sayısı
    "pretrain_save":     "./checkpoints/swin_t_plantclef_pretrained.pth",

    # Faz 2 — Fine-tune (multi-label)
    "finetune_classes":  105,
    "finetune_epochs":   30,
    "finetune_lr":       5e-5,   # pre-train'den düşük
    "freeze_epochs":     5,      # ilk N epoch backbone dondur
    "ml_threshold":      0.5,    # multi-label karar eşiği
    "finetune_save":     "./checkpoints/swin_t_multilabel_best.pth",

    "weight_decay":      1e-4,
    "label_smoothing":   0.1,    # pre-train için
}

os.makedirs("./checkpoints", exist_ok=True)
os.makedirs("./results", exist_ok=True)
print("Konfigürasyon yüklendi.")
print(f"Pre-train batch : {CONFIG['pretrain_batch']} | Fine-tune batch: {CONFIG['finetune_batch']}")
print(f"Pre-train epoch : {CONFIG['pretrain_epochs']} | Fine-tune epoch: {CONFIG['finetune_epochs']}")

In [ ]:
# ── Veri Dönüşümleri — Boyut Uyumu ────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── Faz 1: PlantCLEF (800px max side → 224x224) ──────────────────────────────
# RandomResizedCrop: 800px görüntüden rastgele bölge kesiyor
# scale=(0.4, 1.0) → geniş alan öğrenimi (yaprak, dal, çiçek vb.)
pretrain_transforms = transforms.Compose([
    transforms.RandomResizedCrop(
        CONFIG["pretrain_img_size"],
        scale=(0.4, 1.0),
        ratio=(0.75, 1.33),
        interpolation=transforms.InterpolationMode.BICUBIC,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Faz 2: Hastalık veri seti (256px → 224x224) ───────────────────────────────
# Önce 256px → 224px resize, sonra augmentation
# Hastalık bölgelerini kaybetmemek için center odaklı crop tercih edilir
finetune_train_transforms = transforms.Compose([
    transforms.Resize((CONFIG["finetune_img_size"] + 32, CONFIG["finetune_img_size"] + 32),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(CONFIG["finetune_img_size"]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
    transforms.RandomGrayscale(p=0.03),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Val/Test — hastalık bölgesini tam merkeze al
finetune_eval_transforms = transforms.Compose([
    transforms.Resize((CONFIG["finetune_img_size"], CONFIG["finetune_img_size"]),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("Dönüşüm pipeline hazır.")
print(f"  PlantCLEF (800px) → RandomResizedCrop({CONFIG['pretrain_img_size']}) — scale 0.4-1.0")
print(f"  Hastalık  (256px) → Resize(256) → RandomCrop({CONFIG['finetune_img_size']})")

In [ ]:
# ── DataLoader — Faz 1 & Faz 2 ───────────────────────────────────────────────
plantclef_path = Path(CONFIG["plantclef_dir"])
finetune_path  = Path(CONFIG["finetune_dir"])

# ── Faz 1: PlantCLEF ─────────────────────────────────────────────────────────
pretrain_loader = None
if plantclef_path.exists():
    # ImageFolder: her alt klasör = bir tür
    _pt_root = plantclef_path / "train" if (plantclef_path / "train").exists() else plantclef_path
    pretrain_dataset = datasets.ImageFolder(_pt_root, transform=pretrain_transforms)
    CONFIG["plantclef_classes"] = len(pretrain_dataset.classes)
    pretrain_loader = DataLoader(
        pretrain_dataset,
        batch_size=CONFIG["pretrain_batch"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
        drop_last=True,
    )
    print(f"PlantCLEF: {len(pretrain_dataset):,} görüntü | {CONFIG['plantclef_classes']} tür")
else:
    print("PlantCLEF henüz indirilmedi — Faz 1 atlanacak.")

# ── Faz 2: Hastalık veri seti (multi-label) ───────────────────────────────────
# ImageFolder ile single-label indeksler alınır,
# multi-label etiketleme için soft-label dönüşümü yapılır (bkz. MultiLabelWrapper)
ft_train_raw = datasets.ImageFolder(finetune_path / "train", transform=finetune_train_transforms)
ft_val_raw   = datasets.ImageFolder(finetune_path / "val",   transform=finetune_eval_transforms)
ft_test_raw  = datasets.ImageFolder(finetune_path / "test",  transform=finetune_eval_transforms)

class_names = ft_train_raw.classes
CONFIG["finetune_classes"] = len(class_names)

print(f"\nHastalık veri seti:")
print(f"  Train : {len(ft_train_raw):,} | Val : {len(ft_val_raw):,} | Test : {len(ft_test_raw):,}")
print(f"  Sınıf : {CONFIG['finetune_classes']}")

# ── Multi-label Dataset Wrapper ───────────────────────────────────────────────
# Tek görüntü → multi-label one-hot vektör dönüşümü
# Üretim senaryosu: bir ürün birden fazla hastalık gösterebilir
# Burada eğitim verisi single-label olduğundan one-hot sparse vector kullanılır
class MultiLabelWrapper(Dataset):
    """
    ImageFolder üzerine sarmalayıcı.
    Tek class_idx → num_classes boyutlu float one-hot tensör.
    BCEWithLogitsLoss ile uyumlu.
    """
    def __init__(self, base_dataset, num_classes):
        self.base = base_dataset
        self.num_classes = num_classes

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        target = torch.zeros(self.num_classes, dtype=torch.float32)
        target[label] = 1.0
        return img, target

ft_train = MultiLabelWrapper(ft_train_raw, CONFIG["finetune_classes"])
ft_val   = MultiLabelWrapper(ft_val_raw,   CONFIG["finetune_classes"])
ft_test  = MultiLabelWrapper(ft_test_raw,  CONFIG["finetune_classes"])

ft_train_loader = DataLoader(ft_train, batch_size=CONFIG["finetune_batch"],
                             shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
ft_val_loader   = DataLoader(ft_val,   batch_size=CONFIG["finetune_batch"],
                             shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
ft_test_loader  = DataLoader(ft_test,  batch_size=CONFIG["finetune_batch"],
                             shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

# Sınıf ağırlıkları (dengesizlik) — BCEWithLogitsLoss pos_weight parametresi için
ft_counts = np.array([
    max(len(list((finetune_path / "train" / c).glob("*"))), 1)
    for c in class_names
], dtype=np.float32)
total_samples = ft_counts.sum()
pos_weight = torch.tensor(
    (total_samples - ft_counts) / ft_counts,
    dtype=torch.float32
).to(DEVICE)
print(f"  pos_weight shape: {pos_weight.shape} — min: {pos_weight.min():.2f} max: {pos_weight.max():.2f}")

In [ ]:
# ── Model — Swin-T (Pre-train → Multi-label Fine-tune) ───────────────────────
class BotanixSwinT(nn.Module):
    """
    Swin Transformer Tiny tabanlı ürün hastalık tespit modeli.

    Faz 1 (pre-train):  num_classes=7806, head=Linear(768, 7806)
    Faz 2 (fine-tune):  replace_head(105) → Linear(768, 105) — BCEWithLogitsLoss

    Mobil uyumluluk:
    - Swin-T: ~28M parametre, ~110MB
    - TorchScript uyumlu (dynamic control flow yok)
    - INT8 quantization sonrası ~28MB
    """
    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        # timm Swin-T: global_pool="" → spatial feature map döner
        self.backbone = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",   # [B, 768] çıkış
        )
        feat_dim = self.backbone.num_features  # 768

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(p=0.3),
            nn.Linear(feat_dim, num_classes),
        )
        self._init_head()

    def _init_head(self):
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def replace_head(self, num_classes: int):
        """Fine-tune için head'i yeni sınıf sayısıyla değiştirir."""
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(p=0.3),
            nn.Linear(feat_dim, num_classes),
        )
        self._init_head()
        self.head = self.head.to(next(self.backbone.parameters()).device)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)   # [B, 768]
        return self.head(feat)    # [B, num_classes]

# Faz 1 modeli
model = BotanixSwinT(num_classes=CONFIG["plantclef_classes"], pretrained=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Toplam parametre  : {total_params:,}")
print(f"Eğitilebilir      : {trainable:,}")
print(f"Model boyutu (est): {total_params * 4 / 1e6:.1f} MB")

In [ ]:
# ── Faz 1: Optimizer & Scheduler ─────────────────────────────────────────────
pretrain_criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

# Backbone ve head için ayrı LR (layer-wise decay)
pretrain_optimizer = optim.AdamW([
    {"params": model.backbone.parameters(), "lr": CONFIG["pretrain_lr"]},
    {"params": model.head.parameters(),     "lr": CONFIG["pretrain_lr"] * 2},
], weight_decay=CONFIG["weight_decay"])

# Warmup + CosineAnnealing
def warmup_cosine_lambda(epoch, warmup=2, total=10):
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

pretrain_scheduler = optim.lr_scheduler.LambdaLR(
    pretrain_optimizer,
    lr_lambda=lambda e: warmup_cosine_lambda(
        e, CONFIG["pretrain_warmup"], CONFIG["pretrain_epochs"]
    )
)

print(f"Pre-train Loss     : CrossEntropyLoss (label_smoothing={CONFIG['label_smoothing']})")
print(f"Pre-train Optimizer: AdamW (backbone lr={CONFIG['pretrain_lr']}, head lr={CONFIG['pretrain_lr']*2})")
print(f"Pre-train Scheduler: Warmup({CONFIG['pretrain_warmup']}) + CosineAnnealing")

In [ ]:
# ── Yardımcı Fonksiyonlar ─────────────────────────────────────────────────────
def train_one_epoch_sl(model, loader, optimizer, criterion, device):
    """Single-label eğitim (Faz 1 — PlantCLEF)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_sl(model, loader, criterion, device):
    """Single-label değerlendirme (Faz 1)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def train_one_epoch_ml(model, loader, optimizer, criterion, device):
    """Multi-label eğitim (Faz 2 — hastalık tespiti)."""
    model.train()
    total_loss, total = 0.0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(imgs)                      # [B, 105]
        loss = criterion(logits, targets)         # BCEWithLogitsLoss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)
    return total_loss / total

@torch.no_grad()
def evaluate_ml(model, loader, criterion, device, threshold=0.5):
    """Multi-label değerlendirme — F1 (micro) ve loss döner."""
    model.eval()
    total_loss, total = 0.0, 0
    all_preds, all_targets = [], []
    for imgs, targets in loader:
        imgs, targets = imgs.to(device), targets.to(device)
        logits = model(imgs)
        loss = criterion(logits, targets)
        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()
        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())
    all_preds   = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    f1_micro = f1_score(all_targets, all_preds, average="micro", zero_division=0)
    f1_macro = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return total_loss / total, f1_micro, f1_macro

print("Fonksiyonlar hazır.")
print("  train_one_epoch_sl / evaluate_sl  → Faz 1 (CrossEntropy)")
print("  train_one_epoch_ml / evaluate_ml  → Faz 2 (BCEWithLogitsLoss, multi-label)")

In [ ]:
# ── FAZ 1: PlantCLEF Pre-train ────────────────────────────────────────────────
history_p1 = {"train_loss": [], "train_acc": [], "lr": []}

if pretrain_loader is not None:
    print("=" * 65)
    print("FAZ 1 — PlantCLEF Pre-train (Swin-T, Single-label)")
    print(f"Dataset: {len(pretrain_dataset):,} görüntü | {CONFIG['plantclef_classes']} tür")
    print(f"Batch  : {CONFIG['pretrain_batch']} | Epoch: {CONFIG['pretrain_epochs']}")
    print("=" * 65)

    train_start_p1 = time.time()

    for epoch in range(1, CONFIG["pretrain_epochs"] + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch_sl(
            model, pretrain_loader, pretrain_optimizer, pretrain_criterion, DEVICE
        )
        current_lr = pretrain_optimizer.param_groups[0]["lr"]
        pretrain_scheduler.step()

        history_p1["train_loss"].append(tr_loss)
        history_p1["train_acc"].append(tr_acc)
        history_p1["lr"].append(current_lr)

        elapsed = time.time() - t0
        eta_min = (CONFIG["pretrain_epochs"] - epoch) * elapsed / 60
        print(f"[Pre-train] Epoch [{epoch:02d}/{CONFIG['pretrain_epochs']}] "
              f"Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
              f"LR: {current_lr:.2e} | {elapsed:.0f}s | ETA: {eta_min:.0f}dk")

    total_pretrain_time = time.time() - train_start_p1
    torch.save(model.state_dict(), CONFIG["pretrain_save"])
    print(f"\nPre-train tamamlandı: {total_pretrain_time/60:.1f} dakika")
    print(f"Checkpoint: {CONFIG['pretrain_save']}")

    # Pre-train loss grafiği
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history_p1["train_loss"], marker="o", color="#1565C0")
    axes[0].set_title("Faz 1 — PlantCLEF Train Loss")
    axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)
    axes[1].plot(history_p1["train_acc"], marker="o", color="#2E7D32")
    axes[1].set_title("Faz 1 — PlantCLEF Train Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)
    plt.suptitle("Swin-T PlantCLEF Pre-train")
    plt.tight_layout()
    plt.savefig("./results/swin_t_pretrain_history.png", dpi=150)
    plt.show()
else:
    print("PlantCLEF bulunamadı — Faz 1 atlanıyor.")
    print(f"Mevcut checkpoint varsa: {CONFIG['pretrain_save']}")
    total_pretrain_time = 0

In [ ]:
# ── FAZ 2: Fine-tune — Multi-label Ürün Hastalık Tespiti ─────────────────────
print("=" * 65)
print("FAZ 2 — Multi-label Fine-tune (105 Hastalık Sınıfı)")
print("=" * 65)

# Pre-train checkpoint yükle
ckpt = Path(CONFIG["pretrain_save"])
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print(f"Pre-train ağırlıkları yüklendi: {ckpt}")
else:
    print("Pre-train checkpoint bulunamadı — ImageNet ağırlıklarıyla devam ediliyor.")

# Head'i multi-label için değiştir (105 sınıf)
model.replace_head(num_classes=CONFIG["finetune_classes"])
model = model.to(DEVICE)

# Backbone dondur — ilk freeze_epochs boyunca sadece head eğitilir
model.freeze_backbone()
print(f"Backbone donduruldu (ilk {CONFIG['freeze_epochs']} epoch)")

# BCEWithLogitsLoss — multi-label için standart
ft_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

ft_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["finetune_lr"],
    weight_decay=CONFIG["weight_decay"],
)
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer,
    T_max=CONFIG["finetune_epochs"],
    eta_min=1e-7,
)

history_ft = {
    "train_loss": [], "val_loss": [],
    "val_f1_micro": [], "val_f1_macro": [], "lr": []
}
best_val_f1 = 0.0
train_start_ft = time.time()

for epoch in range(1, CONFIG["finetune_epochs"] + 1):

    # Backbone açma zamanı
    if epoch == CONFIG["freeze_epochs"] + 1:
        model.unfreeze_backbone()
        # Backbone için çok daha düşük LR
        ft_optimizer = optim.AdamW([
            {"params": model.backbone.parameters(), "lr": CONFIG["finetune_lr"] * 0.05},
            {"params": model.head.parameters(),     "lr": CONFIG["finetune_lr"]},
        ], weight_decay=CONFIG["weight_decay"])
        ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            ft_optimizer,
            T_max=CONFIG["finetune_epochs"] - CONFIG["freeze_epochs"],
            eta_min=1e-7,
        )
        print(f"\nEpoch {epoch}: Backbone açıldı "
              f"(backbone lr={CONFIG['finetune_lr']*0.05:.2e}, "
              f"head lr={CONFIG['finetune_lr']:.2e})")

    t0 = time.time()
    tr_loss = train_one_epoch_ml(
        model, ft_train_loader, ft_optimizer, ft_criterion, DEVICE
    )
    vl_loss, vl_f1_micro, vl_f1_macro = evaluate_ml(
        model, ft_val_loader, ft_criterion, DEVICE, CONFIG["ml_threshold"]
    )
    current_lr = ft_optimizer.param_groups[0]["lr"]
    ft_scheduler.step()

    history_ft["train_loss"].append(tr_loss)
    history_ft["val_loss"].append(vl_loss)
    history_ft["val_f1_micro"].append(vl_f1_micro)
    history_ft["val_f1_macro"].append(vl_f1_macro)
    history_ft["lr"].append(current_lr)

    if vl_f1_micro > best_val_f1:
        best_val_f1 = vl_f1_micro
        torch.save(model.state_dict(), CONFIG["finetune_save"])

    frozen = "❄" if epoch <= CONFIG["freeze_epochs"] else "🔥"
    print(f"[Fine-tune {frozen}] Epoch [{epoch:02d}/{CONFIG['finetune_epochs']}] "
          f"Loss: {tr_loss:.4f} | "
          f"Val Loss: {vl_loss:.4f} F1-micro: {vl_f1_micro:.4f} F1-macro: {vl_f1_macro:.4f} | "
          f"LR: {current_lr:.2e} | {time.time()-t0:.1f}s")

total_finetune_time = time.time() - train_start_ft
print(f"\nFine-tune tamamlandı: {total_finetune_time/60:.1f} dakika")
print(f"En iyi Val F1-micro : {best_val_f1:.4f}")

In [ ]:
# ── Eğitim Grafikleri ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_ft["train_loss"], label="Train", color="#E53935")
axes[0].plot(history_ft["val_loss"],   label="Val",   color="#1E88E5")
axes[0].axvline(x=CONFIG["freeze_epochs"]-1, color="gray", ls="--", label="Backbone açıldı")
axes[0].set_title("Fine-tune Loss (BCE)"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_ft["val_f1_micro"], label="F1-micro", color="#43A047")
axes[1].plot(history_ft["val_f1_macro"], label="F1-macro", color="#FB8C00")
axes[1].axvline(x=CONFIG["freeze_epochs"]-1, color="gray", ls="--", label="Backbone açıldı")
axes[1].set_title("Val F1 Score (Multi-label)"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history_ft["lr"], color="#8E24AA", marker=".")
axes[2].set_title("Learning Rate"); axes[2].set_yscale("log"); axes[2].grid(alpha=0.3)

plt.suptitle("Botanix Swin-T — Multi-label Fine-tune Training History")
plt.tight_layout()
plt.savefig("./results/swin_t_multilabel_training.png", dpi=150)
plt.show()

In [ ]:
# ── Test Değerlendirmesi — Multi-label ────────────────────────────────────────
model.load_state_dict(torch.load(CONFIG["finetune_save"], map_location=DEVICE))
model.eval()

all_probs_list, all_targets_list = [], []
infer_start = time.time()

with torch.no_grad():
    for imgs, targets in ft_test_loader:
        logits = model(imgs.to(DEVICE))
        probs  = torch.sigmoid(logits).cpu()
        all_probs_list.append(probs)
        all_targets_list.append(targets)

infer_time   = time.time() - infer_start
all_probs    = torch.cat(all_probs_list).numpy()    # [N, 105]
all_targets  = torch.cat(all_targets_list).numpy()  # [N, 105]

# Eşik bazlı ikili tahmin
all_preds_bin = (all_probs >= CONFIG["ml_threshold"]).astype(float)

# Metrikler
f1_micro  = f1_score(all_targets, all_preds_bin, average="micro",    zero_division=0)
f1_macro  = f1_score(all_targets, all_preds_bin, average="macro",    zero_division=0)
f1_samp   = f1_score(all_targets, all_preds_bin, average="samples",  zero_division=0)
prec_micro = precision_score(all_targets, all_preds_bin, average="micro", zero_division=0)
rec_micro  = recall_score(all_targets, all_preds_bin, average="micro",  zero_division=0)

# mAP (mean Average Precision)
mAP = average_precision_score(all_targets, all_probs, average="macro")

n_test = len(ft_test_raw)
print(f"\n{'='*50}")
print(f"Test Sonuçları — Multi-label (threshold={CONFIG['ml_threshold']})")
print(f"{'='*50}")
print(f"F1-micro   : {f1_micro:.4f}")
print(f"F1-macro   : {f1_macro:.4f}")
print(f"F1-samples : {f1_samp:.4f}")
print(f"Precision  : {prec_micro:.4f}")
print(f"Recall     : {rec_micro:.4f}")
print(f"mAP        : {mAP:.4f}")
print(f"Inference  : {infer_time/n_test*1000:.2f} ms/görüntü")

In [ ]:
# ── Kapsamlı Grafiksel Değerlendirme ─────────────────────────────────────────
MODEL_LABEL  = "Botanix Swin-T (PlantCLEF → Multi-label)"
MODEL_PREFIX = "swin_t_multilabel"

# ── 1. Per-Class F1 Score ─────────────────────────────────────────────────────
per_class_f1 = f1_score(all_targets, all_preds_bin, average=None, zero_division=0)
sorted_idx   = np.argsort(per_class_f1)
worst20 = [(class_names[i], per_class_f1[i]) for i in sorted_idx[:20]]
best20  = [(class_names[i], per_class_f1[i]) for i in sorted_idx[-20:][::-1]]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].barh([x[0][:30] for x in worst20], [x[1] for x in worst20], color="#EF5350")
axes[0].set_title(f"{MODEL_LABEL}\nEn Düşük F1 (20 Sınıf)")
axes[0].set_xlabel("F1 Score"); axes[0].set_xlim(0, 1); axes[0].grid(alpha=0.3)
axes[1].barh([x[0][:30] for x in best20],  [x[1] for x in best20],  color="#66BB6A")
axes[1].set_title(f"{MODEL_LABEL}\nEn Yüksek F1 (20 Sınıf)")
axes[1].set_xlabel("F1 Score"); axes[1].set_xlim(0, 1); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_per_class_f1.png", dpi=150)
plt.show()

# ── 2. Precision-Recall Curve (mAP) ──────────────────────────────────────────
from sklearn.metrics import precision_recall_curve
fig, ax = plt.subplots(figsize=(8, 6))
# Top-5 en düşük F1'li sınıf için ayrı PR eğrisi
colors = plt.cm.Set2(np.linspace(0, 1, 5))
for idx_c, color in zip(sorted_idx[:5], colors):
    if all_targets[:, idx_c].sum() > 0:
        prec_c, rec_c, _ = precision_recall_curve(all_targets[:, idx_c], all_probs[:, idx_c])
        ap_c = average_precision_score(all_targets[:, idx_c], all_probs[:, idx_c])
        ax.plot(rec_c, prec_c, color=color, lw=1.5,
                label=f"{class_names[idx_c][:20]} (AP={ap_c:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"{MODEL_LABEL}\nPrecision-Recall (En Zor 5 Sınıf) | mAP={mAP:.4f}")
ax.legend(fontsize=8, loc="upper right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_pr_curve.png", dpi=150)
plt.show()

# ── 3. Makro-Ortalama ROC Eğrisi ─────────────────────────────────────────────
fpr_dict, tpr_dict, roc_dict = {}, {}, {}
for i in range(CONFIG["finetune_classes"]):
    if all_targets[:, i].sum() > 0:
        fpr_dict[i], tpr_dict[i], _ = roc_curve(all_targets[:, i], all_probs[:, i])
        roc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

all_fpr  = np.unique(np.concatenate([fpr_dict[i] for i in roc_dict]))
mean_tpr = np.zeros_like(all_fpr)
for i in roc_dict:
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= len(roc_dict)
macro_auc = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(all_fpr, mean_tpr, color="#1565C0", lw=2,
        label=f"Makro-Ortalama ROC (AUC = {macro_auc:.4f})")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL_LABEL}\nROC Eğrisi (Makro Ortalama)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_roc.png", dpi=150)
plt.show()

# ── 4. Eşik Analizi ──────────────────────────────────────────────────────────
thresholds = np.arange(0.1, 0.9, 0.05)
th_f1_micro = []
for th in thresholds:
    preds_th = (all_probs >= th).astype(float)
    th_f1_micro.append(f1_score(all_targets, preds_th, average="micro", zero_division=0))

best_th  = thresholds[np.argmax(th_f1_micro)]
best_f1_th = max(th_f1_micro)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, th_f1_micro, marker="o", color="#00897B", lw=2)
ax.axvline(x=best_th, color="red", ls="--", label=f"En iyi eşik: {best_th:.2f} (F1={best_f1_th:.4f})")
ax.set_xlabel("Karar Eşiği"); ax.set_ylabel("F1-micro")
ax.set_title(f"{MODEL_LABEL} — Eşik Analizi")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_threshold.png", dpi=150)
plt.show()
print(f"En iyi eşik: {best_th:.2f} → F1-micro: {best_f1_th:.4f}")

In [ ]:
# ── Mobil Export Hazırlığı ────────────────────────────────────────────────────
print("Mobil export hazırlığı başlıyor...")

model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)

# ── 1. TorchScript (iOS Core ML & Android TorchMobile) ───────────────────────
try:
    scripted = torch.jit.trace(model, dummy_input)
    scripted.save("./checkpoints/swin_t_multilabel_scripted.pt")
    print("TorchScript kaydedildi: ./checkpoints/swin_t_multilabel_scripted.pt")
except Exception as e:
    print(f"TorchScript hatası: {e}")

# ── 2. ONNX Export (TFLite/TensorRT/web) ─────────────────────────────────────
try:
    torch.onnx.export(
        model,
        dummy_input,
        "./checkpoints/swin_t_multilabel.onnx",
        opset_version=17,
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={"image": {0: "batch_size"}, "logits": {0: "batch_size"}},
    )
    print("ONNX kaydedildi: ./checkpoints/swin_t_multilabel.onnx")
except Exception as e:
    print(f"ONNX hatası: {e}")

# ── 3. Model Boyutu & Inference Süresi ───────────────────────────────────────
import os
pth_size = os.path.getsize(CONFIG["finetune_save"]) / 1e6
print(f"\n.pth boyutu  : {pth_size:.1f} MB")

# GPU inference benchmark
model.eval()
warmup_runs = 20
bench_runs  = 100
bench_input = torch.randn(1, 3, 224, 224).to(DEVICE)

with torch.no_grad():
    for _ in range(warmup_runs):
        _ = model(bench_input)
    torch.cuda.synchronize()
    t_start = time.time()
    for _ in range(bench_runs):
        _ = model(bench_input)
    torch.cuda.synchronize()
    gpu_inf_ms = (time.time() - t_start) / bench_runs * 1000

print(f"GPU inference : {gpu_inf_ms:.2f} ms/görüntü")
print(f"\nMobil hedef  : ~15-30ms (INT8 quantization sonrası)")

In [ ]:
# ── Sonuçları Kaydet ──────────────────────────────────────────────────────────
results = {
    "model": "Botanix Swin-T (PlantCLEF Pre-train → Multi-label Fine-tune)",
    "task": "multi-label",
    "f1_micro":    round(f1_micro,  4),
    "f1_macro":    round(f1_macro,  4),
    "f1_samples":  round(f1_samp,   4),
    "precision":   round(prec_micro, 4),
    "recall":      round(rec_micro,  4),
    "mAP":         round(mAP,        4),
    "roc_auc_macro": round(macro_auc, 4),
    "best_threshold": round(float(best_th), 2),
    "best_f1_at_best_threshold": round(float(best_f1_th), 4),
    "pretrain_time_min":  round(total_pretrain_time / 60, 2),
    "finetune_time_min":  round(total_finetune_time / 60, 2),
    "inference_ms_per_image": round(infer_time / len(ft_test_raw) * 1000, 4),
    "gpu_inference_ms": round(gpu_inf_ms, 2),
    "total_params": sum(p.numel() for p in model.parameters()),
    "config": CONFIG,
}
with open("./results/swin_t_multilabel_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## Mobil Uyumluluk Notu — Botanix Swin-T Multi-label

| Özellik | Değer |
|---------|-------|
| Mimari | Swin Transformer Tiny |
| Parametre sayısı | ~28M |
| Model boyutu (.pth) | ~110 MB |
| Model boyutu (INT8) | ~28 MB |
| GPU Inference | ~5-8 ms/görüntü |
| Mobil hedef inference | ~15-30 ms |
| Mobil uygunluğu | ✅ Orta-Yüksek |
| Çıkış türü | Multi-label sigmoid (105 sınıf) |

**Export dosyaları:**
- `swin_t_multilabel_scripted.pt` → iOS (Core ML Tools) / Android (TorchMobile)
- `swin_t_multilabel.onnx` → TFLite (Android), TensorRT, web

**Üretim kullanımı:**
```python
# Mobil tarafında sigmoid + threshold uygulanmalı
probs = torch.sigmoid(model(image))          # [1, 105]
detected = (probs >= 0.5).nonzero().squeeze()  # aktif hastalıklar
detected_names = [class_names[i] for i in detected]
```

**INT8 Quantization (opsiyonel, boyutu 4x küçültür):**
```python
import torch.quantization
model_int8 = torch.quantization.quantize_dynamic(
    model.cpu(), {nn.Linear}, dtype=torch.qint8
)
torch.jit.save(torch.jit.script(model_int8), "swin_t_int8.pt")
```